# 3. Is it a 7? Real handwriting

Handwritten digits, 8 x 8 pixels each (64 inputs). Both methods learn to answer 7 or not 7, then are tested on digits they never saw.

Part of Kyle Robin's DCS 340 Project II on NEAT (Stanley & Miikkulainen, 2002). Run the cells top to bottom with **Shift + Enter**.

In [ ]:
# Run this first: installs NEAT (takes a few seconds)
!pip install -q neat-python

In [ ]:
# Setup: imports and shared helper code. Run this cell first.
import os, time, math, tempfile, contextlib, io, warnings
import numpy as np
import matplotlib.pyplot as plt
import neat

warnings.filterwarnings("ignore")
SEED = 0  # change this to see how much results depend on luck


# ---------- Gradient descent: a small network written from scratch ----------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


class GDNet:
    """A network with ONE hidden layer whose size YOU choose.
    It learns with plain gradient descent + backpropagation."""

    def __init__(self, n_inputs=2, n_hidden=4, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 1, (n_inputs, n_hidden))  # input -> hidden weights
        self.b1 = np.zeros(n_hidden)                       # hidden biases
        self.W2 = rng.normal(0, 1, (n_hidden, 1))          # hidden -> output weights
        self.b2 = np.zeros(1)                              # output bias

    def forward(self, X):
        """Forward pass: weighted sum + sigmoid, layer by layer."""
        self.h = sigmoid(X @ self.W1 + self.b1)
        self.out = sigmoid(self.h @ self.W2 + self.b2)
        return self.out[:, 0]

    def train(self, X, y, lr=2.0, epochs=5000):
        """Gradient descent on mean squared error. Returns the loss at every step."""
        losses = []
        for _ in range(epochs):
            p = self.forward(X)
            err = p - y
            losses.append(np.mean(err ** 2))
            # Backpropagation: pass the error backward with the chain rule
            d_out = (2 * err / len(y) * p * (1 - p))[:, None]
            gW2, gb2 = self.h.T @ d_out, d_out.sum(0)
            d_h = d_out @ self.W2.T * self.h * (1 - self.h)
            gW1, gb1 = X.T @ d_h, d_h.sum(0)
            # Step downhill: w_new = w_old - learning_rate * slope
            self.W2 -= lr * gW2; self.b2 -= lr * gb2
            self.W1 -= lr * gW1; self.b1 -= lr * gb1
        return losses

    def n_params(self):
        return self.W1.size + self.b1.size + self.W2.size + self.b2.size

    def predict(self, X):
        return self.forward(np.asarray(X, float))


# ---------- NEAT: settings and a runner ----------
NEAT_SETTINGS = {
    "num_inputs": 2, "pop_size": 150, "fitness_threshold": 3.9,
    "node_add_prob": 0.2, "conn_add_prob": 0.5, "weight_mutate_power": 0.5,
    "activation_options": "sigmoid", "activation_mutate_rate": 0.0,
    "compatibility_threshold": 3.0,
}

NEAT_TEMPLATE = """
[NEAT]
fitness_criterion = max
fitness_threshold = {fitness_threshold}
pop_size = {pop_size}
reset_on_extinction = True
no_fitness_termination = False
[DefaultGenome]
num_inputs = {num_inputs}
num_hidden = 0
num_outputs = 1
initial_connection = full
feed_forward = True
activation_default = sigmoid
activation_mutate_rate = {activation_mutate_rate}
activation_options = {activation_options}
aggregation_default = sum
aggregation_mutate_rate = 0.0
aggregation_options = sum
bias_init_mean = 0.0
bias_init_stdev = 1.0
bias_max_value = 30.0
bias_min_value = -30.0
bias_mutate_power = {weight_mutate_power}
bias_mutate_rate = 0.7
bias_replace_rate = 0.1
response_init_mean = 1.0
response_init_stdev = 0.0
response_max_value = 30.0
response_min_value = -30.0
response_mutate_power = 0.0
response_mutate_rate = 0.0
response_replace_rate = 0.0
weight_init_mean = 0.0
weight_init_stdev = 1.0
weight_max_value = 30
weight_min_value = -30
weight_mutate_power = {weight_mutate_power}
weight_mutate_rate = 0.8
weight_replace_rate = 0.1
enabled_default = True
enabled_mutate_rate = 0.01
conn_add_prob = {conn_add_prob}
conn_delete_prob = 0.5
node_add_prob = {node_add_prob}
node_delete_prob = 0.2
compatibility_disjoint_coefficient = 1.0
compatibility_weight_coefficient = 0.5
[DefaultSpeciesSet]
compatibility_threshold = {compatibility_threshold}
[DefaultStagnation]
species_fitness_func = max
max_stagnation = 20
species_elitism = 2
[DefaultReproduction]
elitism = 2
survival_threshold = 0.2
"""


def run_neat(fitness_fn, changes=None, generations=300, seed=0):
    """Evolve networks with NEAT. fitness_fn(net) -> a score (higher is better).
    Returns (best genome, its network, config, stats, seconds)."""
    settings = dict(NEAT_SETTINGS, **(changes or {}))
    path = os.path.join(tempfile.gettempdir(), "neat_compare.ini")
    with open(path, "w") as f:
        f.write(NEAT_TEMPLATE.format(**settings))
    config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                         neat.DefaultSpeciesSet, neat.DefaultStagnation, path)

    def eval_genomes(genomes, config):
        for _, g in genomes:
            g.fitness = fitness_fn(neat.nn.FeedForwardNetwork.create(g, config))

    pop = neat.Population(config, seed=seed)
    stats = neat.StatisticsReporter()
    pop.add_reporter(stats)
    start = time.time()
    with contextlib.redirect_stdout(io.StringIO()):
        winner = pop.run(eval_genomes, generations)
    net = neat.nn.FeedForwardNetwork.create(winner, config)
    return winner, net, config, stats, time.time() - start


def neat_size(winner, config):
    """(hidden neurons, enabled connections, total parameters = weights + biases)"""
    hidden = sum(1 for k in winner.nodes if k not in config.genome_config.output_keys)
    conns = sum(c.enabled for c in winner.connections.values())
    return hidden, conns, conns + len(winner.nodes)


def neat_predict(net, X):
    return np.array([net.activate(x)[0] for x in X])


def accuracy(pred, y):
    return np.mean((pred > 0.5) == y)

print("Ready.")


In [ ]:
# Load ~1,800 handwritten digits that come with scikit-learn (8 x 8 pixels each)
from sklearn.datasets import load_digits

digits = load_digits()
DX = digits.data / 16.0                    # 64 pixel brightnesses per image, scaled 0 to 1
DY = (digits.target == 7).astype(float)    # 1 if the digit is a 7, else 0

rng = np.random.default_rng(SEED)
order = rng.permutation(len(DY))
train, test = order[:1200], order[1200:]   # test digits are never seen during training

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, i in zip(axes.ravel(), np.where(DY == 1)[0][:16]):
    ax.imshow(digits.images[i], cmap="gray_r")
    ax.axis("off")
fig.suptitle("16 of the handwritten 7s")
plt.show()
print(f"{int(DY.sum())} sevens out of {len(DY)} digits. Training on {len(train)}, testing on {len(test)}.")

In [ ]:
def balanced_accuracy(pred, y):
    """Average of (% of 7s caught) and (% of non-7s correctly rejected).
    Plain accuracy would mislead: answering 'not a 7' every time already scores 90%."""
    p = pred > 0.5
    return 0.5 * (np.mean(p[y == 1]) + np.mean(~p[y == 0]))

# Gradient descent: 64 inputs, 8 hidden neurons (our choice), 1 output
start = time.time()
gd7 = GDNet(n_inputs=64, n_hidden=8, seed=SEED)
gd7.train(DX[train], DY[train], lr=2.0, epochs=2000)
gd7_secs = time.time() - start

# NEAT: fitness = balanced accuracy on the training 7s plus twice as many non-7s
# (a smaller, balanced sample keeps each generation fast)
pos = train[DY[train] == 1]
neg = train[DY[train] == 0][: 2 * len(pos)]
fit_idx = np.r_[pos, neg]

def seven_fitness(net):
    return balanced_accuracy(neat_predict(net, DX[fit_idx]), DY[fit_idx])

w7, neat7, cfg7, stats7, neat7_secs = run_neat(
    seven_fitness, {"num_inputs": 64, "fitness_threshold": 0.97}, generations=100, seed=SEED)
h7, c7, p7 = neat_size(w7, cfg7)

gd_test = gd7.predict(DX[test])
neat_test = neat_predict(neat7, DX[test])

print(f"{'method':18s} {'test score':>10s} {'hidden':>7s} {'params':>7s} {'seconds':>8s}")
print(f"{'Gradient descent':18s} {balanced_accuracy(gd_test, DY[test]):10.1%} {8:7d} {gd7.n_params():7d} {gd7_secs:8.1f}")
print(f"{'NEAT':18s} {balanced_accuracy(neat_test, DY[test]):10.1%} {h7:7d} {p7:7d} {neat7_secs:8.1f}")
print(f"\nNEAT finished in {len(stats7.most_fit_genomes)} generations.")

In [ ]:
# Which test digits did NEAT get wrong?
wrong = np.where((neat_test > 0.5) != (DY[test] == 1))[0]
print(f"NEAT got {len(wrong)} of {len(test)} test digits wrong.")
if len(wrong):
    n = min(8, len(wrong))
    fig, axes = plt.subplots(1, n, figsize=(1.6 * n, 2.2), squeeze=False)
    for ax, k in zip(axes[0], wrong[:n]):
        i = test[k]
        ax.imshow(digits.images[i], cmap="gray_r")
        ax.set_title("missed 7" if DY[i] == 1 else f"said 7\n(was {digits.target[i]})", fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

**Your notes:**

- 
